In [44]:
import pandas as pd
import numpy as np
from scripts.Roni.ASOptimizer_IDO1.IDO_Seq import IDO1_sequence
from asodesigner.util import get_antisense
from scripts.data_genertion.data_handling import populate_features, get_populate_fold
from asodesigner.features.RNaseH_features import rnaseh1_dict, compute_rnaseh1_score
from xgboost import XGBRanker


functions to find ASO coordinate on the transcript

In [45]:
def dna_to_dna_reverse_complement(seq: str) -> str:
    seq = seq.upper()
    translation_table = str.maketrans("ATGC", "TACG")
    # Translate and reverse
    return seq.translate(translation_table)[::-1]

In [46]:
def find_aso_binding_positions(aso_seq, mrna_seq):
    target = dna_to_dna_reverse_complement(aso_seq)
    idx = mrna_seq.find(target)
    if idx == -1:
        return None
    return int(idx), idx/len(mrna_seq)

Retrieve the data

In [47]:
exp_data = pd.read_csv("IDO1_exp_inhibition.csv")
exp_data = exp_data[["ASO", "Sequence", "Length"]]

exp_data[["sense_start", "normalized_start"]] = exp_data["Sequence"].apply(
    lambda s: pd.Series(find_aso_binding_positions(s, IDO1_sequence))
)
exp_data = exp_data.dropna(subset=["sense_start"])


In [48]:
print(exp_data)

        ASO           Sequence  Length  sense_start  normalized_start
2   A06055H  CCGCAGGCCAGCATCAC      17       1002.0          0.541915
3   A06049H  ACAAAACGTCCATGTTC      17        469.0          0.253651
4   A06037H   CAGGACGTCAAAGCAC      16        844.0          0.456463
5   A06017H    AGGACGTCAAAGCAC      15        844.0          0.456463
6   A06048H  GTTGGCAGTAAGGAACA      17        355.0          0.191996
..      ...                ...     ...          ...               ...
69  A06001H     GGCGCTGTGACTTG      14        250.0          0.135208
70  A06030H   AGGCGCTGTGACTTGT      16        249.0          0.134667
71  A06045H  AGGCGCTGTGACTTGTG      17        248.0          0.134127
72  A06029H   GGCGCTGTGACTTGTG      16        248.0          0.134127
73  A06009H    GGCGCTGTGACTTGT      15        249.0          0.134667

[72 rows x 5 columns]
